# Setup

In [ ]:
!pip install -qU accelerate chromadb sentence-transformers PyMuPDF bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 3.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 342.1/342.1 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.1/611.1 kB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 80.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.0/20.0 MB 100.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 29.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.6/278.6 kB 24.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.8/94.8 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 87.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 115.9 MB/s et

Zainstalowane biblioteki to:
- `requests` - umożliwia wykonywanie zapytań HTTP
- `accelerate` - biblioteka ułatwiająca przyspieszenie obliczeń na różnych urządzeniach (CPU/GPU)
- `chromadb` - wektorowa baza danych służąca do przechowywania i wyszukiwania osadzonych wektorów (embeddings)
- `sentence-transformers` - narzędzie do generowania wektorowych reprezentacji tekstu
- `PyMuPDF` - biblioteka do pracy z dokumentami PDF
- `bitsandbytes` - optymalizuje wykorzystanie pamięci dla dużych modeli

In [ ]:
import warnings
import fitz
import json
from sentence_transformers import SentenceTransformer
import chromadb
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TextStreamer
import uuid
import os
import pandas as pd

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
warnings.filterwarnings("ignore")

In [ ]:
class CFG:
    model = 'speakleash/Bielik-11B-v2.2-Instruct'
    device = 'cuda'
    muhdoc = "/content/ai_act_pl.pdf"
    sentence = 'ipipan/silver-retriever-base-v1.1'
    datadir = '/content/data'
    temperature = 0.1
    max_tokens = 1000
    top_k = 200
    top_p = 1
    dtype = torch.bfloat16

In [ ]:
if not os.path.exists(CFG.datadir):
    os.makedirs(CFG.datadir)

# Przygotowania

In [ ]:
# embeddings
document = fitz.open(CFG.muhdoc)
pages = []

for page_num in range(len(document)):
    page = document.load_page(page_num)
    page_text = page.get_text()
    pages.append({"page_num": page_num, "text": page_text})

with open('myfile.json', "w") as file:
    json.dump(pages, file, indent=4, ensure_ascii=False)

In [ ]:
texts = []
for page in pages:
    texts.append(page["text"])

embedding_model = SentenceTransformer(CFG.sentence)
embeddings = embedding_model.encode(texts)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/117 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/9.35k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/789 [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/907k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/556k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.30M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/144 [00:00<?, ?B/s]

1_Pooling%2Fconfig.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Te linie kodu wykonują kluczowy proces wektoryzacji tekstu, przekształcając zawartość dokumentu PDF na reprezentacje numeryczne, które można efektywnie przeszukiwać. Przejdźmy przez ten proces krok po kroku.

Na początku tworzona jest pusta lista, która będzie przechowywać tekst ze wszystkich stron dokumentu:
```python
texts = []
```

Następnie kod iteruje przez wszystkie strony dokumentu, pobierając tekst z każdej strony i dodając go do listy:
```python
for page in pages:
    texts.append(page["text"])
```

W tym momencie zmienna `texts` zawiera listę napisów, gdzie każdy element reprezentuje pełną zawartość tekstową jednej strony dokumentu. Struktura ta upraszcza dalsze przetwarzanie, ponieważ pozwala na operowanie na tekście bez konieczności odnoszenia się do numerów stron.

Kolejny krok to inicjalizacja modelu do tworzenia embeddingów (wektorowych reprezentacji tekstu):
```python
embedding_model = SentenceTransformer(CFG.sentence)
```

`SentenceTransformer` to klasa z biblioteki `sentence-transformers`, która implementuje modele transformerowe specjalizujące się w tworzeniu embeddingów zdań. Parametr `CFG.sentence` przekazuje identyfikator konkretnego modelu, który wcześniej został zdefiniowany jako `ipipan/silver-retriever-base-v1.1` – polski model stworzony przez Instytut Podstaw Informatyki PAN, zoptymalizowany do zadań wyszukiwania informacji.

Wreszcie, model zostaje użyty do przekształcenia tekstów w embeddingi:
```python
embeddings = embedding_model.encode(texts)
```

Metoda `encode()` przetwarza każdy element listy `texts` (czyli tekst każdej strony) na wektor liczbowy reprezentujący znaczenie semantyczne tego tekstu. Wynikowa zmienna `embeddings` to tablica numeryczna (prawdopodobnie typu numpy.ndarray), gdzie każdy wiersz odpowiada jednemu fragmentowi tekstu.

Te wektory mają niezwykłe właściwości, które czynią je potężnym narzędziem w przetwarzaniu języka naturalnego:

1. **Podobieństwo semantyczne**: Teksty o podobnym znaczeniu mają podobne wektory (bliskie sobie w przestrzeni wektorowej), nawet jeśli używają różnych słów.

2. **Zachowanie kontekstu**: Wektory uwzględniają kontekst słów, więc np. "bank" w kontekście finansowym będzie miał inną reprezentację niż "bank" w kontekście geograficznym.

3. **Wymiarowość**: Typowo takie wektory mają kilkaset wymiarów (np. 768), co pozwala na uchwycenie subtelnych niuansów znaczeniowych.

W kontekście całego systemu ten etap jest fundamentalny – przekształca surowy tekst w formę matematyczną, która pozwala na:

- Efektywne porównywanie fragmentów tekstu pod względem podobieństwa znaczeniowego
- Wyszukiwanie fragmentów tekstu najbardziej odpowiadających zapytaniu użytkownika
- Zapisanie wektorów w bazie danych wektorowych (jak ChromaDB), co umożliwia szybkie przeszukiwanie dużych zbiorów dokumentów


# Baza

In [ ]:

client = chromadb.PersistentClient(path = CFG.datadir)
collection = client.get_or_create_collection(
        name="ha_naive_rag", metadata={"hnsw:space": "cosine"}
)

collection.add(
        documents=texts,
        embeddings=embeddings,
        ids=[str(i) for i in range(len(texts))]
)

Ten fragment kodu zajmuje się tworzeniem i zapełnianiem bazy wektorowej, która stanowi serce systemu wyszukiwania informacji. Przyjrzyjmy się, jak działa każda linia tego kodu.

Najpierw tworzony jest klient bazy danych wektorowych ChromaDB:

```python
client = chromadb.PersistentClient(path = CFG.datadir)
```

ChromaDB to baza danych specjalizująca się w przechowywaniu i wyszukiwaniu wektorów (embedingów). Użycie `PersistentClient` oznacza, że dane będą zapisywane na dysku pod ścieżką zdefiniowaną w `CFG.datadir` (którą wcześniej określono jako '/content/data'). Dzięki temu dane nie znikną po zakończeniu sesji i mogą być wykorzystane w przyszłości bez konieczności ponownego przetwarzania dokumentu.

W następnym kroku tworzona jest (lub pobierana, jeśli już istnieje) kolekcja w bazie danych:

```python
collection = client.get_or_create_collection(
        name="ha_naive_rag", metadata={"hnsw:space": "cosine"}
)
```

Kolekcja to kontener na wektory - coś w rodzaju tabeli w tradycyjnej bazie danych. Nazwa "ha_naive_rag" wskazuje, że jest to prosta implementacja systemu RAG (Retrieval-Augmented Generation).

Parametr `metadata={"hnsw:space": "cosine"}` jest szczególnie interesujący. Określa on, że do indeksowania i wyszukiwania wektorów będzie używany algorytm HNSW (Hierarchical Navigable Small World), który jest bardzo wydajny dla dużych zbiorów danych. "cosine" oznacza, że do mierzenia podobieństwa między wektorami będzie używana miara kosinusowa - standardowa metoda w wyszukiwaniu semantycznym. Miara kosinusowa określa podobieństwo jako kosinus kąta między dwoma wektorami, ignorując ich długość, co jest pożądane przy porównywaniu znaczenia tekstów.

Ostatni krok to dodanie dokumentów, ich wektorów i identyfikatorów do kolekcji:

```python
collection.add(
        documents=texts,
        embeddings=embeddings,
        ids=[str(i) for i in range(len(texts))]
)
```

W tym wywołaniu przekazywane są trzy parametry:
1. `documents=texts` - lista tekstów stron, które chcemy przechować
2. `embeddings=embeddings` - lista wektorów odpowiadających tym tekstom
3. `ids=[str(i) for i in range(len(texts))]` - lista unikalnych identyfikatorów dla każdego dokumentu

Wyrażenie `[str(i) for i in range(len(texts))]` to tzw. "list comprehension" (wyrażenie listowe) w Pythonie, które tworzy listę identyfikatorów będących ciągami znaków ("0", "1", "2", itd.), po jednym dla każdej strony dokumentu. Każdy identyfikator musi być unikalny i w formacie tekstowym.

Po wykonaniu tego kodu, nasza baza wektorowa zawiera teksty wszystkich stron dokumentu wraz z ich reprezentacjami wektorowymi. Od teraz możemy szybko wyszukiwać strony najbardziej podobne semantycznie do dowolnego zapytania.

Jest to kluczowy element architektury RAG. Gdy użytkownik zada pytanie, system:
1. Przekształci to pytanie na wektor przy użyciu tego samego modelu `embedding_model`
2. Użyje bazy ChromaDB do znalezienia stron dokumentu o najbardziej podobnych wektorach
3. Przekaże te strony jako kontekst do modelu językowego, który na ich podstawie wygeneruje odpowiedź

Taka architektura łączy zalety modeli językowych (zdolność do rozumienia i generowania tekstu) z precyzją wyszukiwania w konkretnych dokumentach. Dzięki temu system może odpowiadać na pytania dotyczące konkretnego dokumentu bez polegania na ogólnej wiedzy modelu, co zwiększa dokładność i wiarygodność odpowiedzi.

Co więcej, ponieważ używamy przestrzeni kosinusowej, system może znajdować semantycznie powiązane fragmenty, nawet jeśli pytanie używa innych sformułowań niż te zawarte w dokumencie, co daje mu elastyczność niedostępną dla prostych systemów wyszukiwania słów kluczowych.

# Model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CFG.model)
tokenizer.pad_token = tokenizer.eos_token

streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)


tokenizer_config.json:   0%|          | 0.00/27.2k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.82M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/3.49k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/3.44k [00:00<?, ?B/s]

Ten kod konfiguruje tokenizer i streamer dla modelu językowego:

`tokenizer = AutoTokenizer.from_pretrained(CFG.model)` - wczytuje tokenizer odpowiedni dla modelu zdefiniowanego w konfiguracji. Tokenizer zamienia tekst na liczby zrozumiałe dla modelu

`tokenizer.pad_token = tokenizer.eos_token` - ustawia token wypełniający (padding) jako token końca sekwencji. Jest to standardowa praktyka dla modeli bazujących na architekturze LLaMA

`streamer = TextStreamer(tokenizer, skip_prompt=True, skip_special_tokens=True)` - tworzy obiekt do strumieniowego wyświetlania generowanego tekstu:
- skip_prompt=True - pomija wyświetlanie początkowego zapytania
- skip_special_tokens=True - pomija wyświetlanie tokenów specjalnych

Te komponenty są niezbędne do komunikacji z modelem językowym - tokenizer przetwarza tekst wejściowy, a streamer zarządza wyświetlaniem generowanej odpowiedzi.

In [ ]:

quantization_config = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_compute_dtype = CFG.dtype)

model = AutoModelForCausalLM.from_pretrained(CFG.model,
                                             torch_dtype = CFG.dtype,
                                             quantization_config=quantization_config)

config.json:   0%|          | 0.00/598 [00:00<?, ?B/s]

`low_cpu_mem_usage` was None, now default to True since model is quantized.


model.safetensors.index.json:   0%|          | 0.00/37.3k [00:00<?, ?B/s]

model-00001-of-00005.safetensors:   0%|          | 0.00/4.94G [00:00<?, ?B/s]

model-00002-of-00005.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00005.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00004-of-00005.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

model-00005-of-00005.safetensors:   0%|          | 0.00/2.56G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/157 [00:00<?, ?B/s]

Ten kod konfiguruje i wczytuje model językowy z kwantyzacją:

`quantization_config = BitsAndBytesConfig(...)` - tworzy konfigurację kwantyzacji:
- `load_in_4bit=True` - włącza kwantyzację 4-bitową, która redukuje zużycie pamięci
- `bnb_4bit_compute_dtype=torch.bfloat16` - ustawia format obliczeń na bfloat16, który jest kompromisem między dokładnością a wydajnością

`model = AutoModelForCausalLM.from_pretrained(...)` - wczytuje model językowy:
- `CFG.model` - identyfikator modelu
- `torch_dtype = CFG.dtype` - ustawia typ danych zgodnie z konfiguracją
- `quantization_config=quantization_config` - stosuje zdefiniowaną wcześniej kwantyzację

Kwantyzacja 4-bitowa znacząco zmniejsza wymagania pamięciowe modelu przy zachowaniu dobrej jakości odpowiedzi, co pozwala na używanie dużych modeli na standardowym sprzęcie.

# Funkcje

In [ ]:
def create_context(query, top=3):
    embedding = embedding_model.encode([query])

    result = collection.query(
            query_embeddings=embedding,
            n_results=top
    )

    if not result:
        print("Brak kontekstu")
        return None

    documents = result.get("documents", [])

    if documents:
        context = "KONTEKST:\n"
        context = context + documents[0][0] + "\n\n"
        if len(documents) > 1:
            context = context + documents[0][1] + "\n\n"
    else:
        print("Brak dokumentów w wynikach.")
        return None

    # zapis do pliku
    filename = f"{str(uuid.uuid4())[:8]}_context.json"

    with open(filename, "w") as file:
        content = {
                "context": context,
                "query": query
        }
        json.dump(content, file, ensure_ascii=False, indent=4)

    return f"Wyłącznie na podstawie podanego kontekstu odpowiedz zwięźle na pytanie: '{query}'\n{context}"

In [ ]:
def generate(prompt, system=None):
    messages = []

    if system:
        messages.append({"role": "system", "content": system})

    messages.append({"role": "user", "content": prompt})

    input_ids = tokenizer.apply_chat_template(messages, return_tensors="pt")
    model_inputs = input_ids

    model_inputs = input_ids.to(CFG.device)

    # generowanie odpowiedzi
    outputs = model.generate(model_inputs,
                             streamer=streamer,
                             max_new_tokens = CFG.max_tokens,
                             do_sample = True if CFG.temperature else False,
                             temperature = CFG.temperature,
                             top_k = CFG.top_k,
                             top_p = CFG.top_p)

    # zapis do pliku
    filename = f"{str(uuid.uuid4())[:8]}.json"

    with open(filename, "w") as file:
        content = {
                "prompt": messages,
                "output": tokenizer.batch_decode(outputs, skip_special_tokens=False)
        }
        json.dump(content, file, ensure_ascii=False, indent=4)

# Run

In [ ]:
prompt = create_context(query = "Czym są systemy wysokiego ryzyka?", top = 3)
generate(prompt)


Systemy wysokiego ryzyka to systemy sztucznej inteligencji (AI), które ze względu na swoje zastosowanie, funkcje lub potencjalne konsekwencje niosą ze sobą znaczące ryzyko dla zdrowia, bezpieczeństwa lub praw podstawowych osób fizycznych. Takie systemy wymagają szczególnych środków technicznych i organizacyjnych, aby zapewnić ich solidność, cyberbezpieczeństwo oraz zgodność z wymogami prawnymi, w tym z zasadami etycznymi i ochroną danych osobowych.

W kontekście podanym, systemy wysokiego ryzyka muszą być odporne na szkodliwe zachowania, zapewniać mechanizmy bezpiecznego przerwania działania w przypadku nieprawidłowości oraz posiadać odpowiednie środki cyberbezpieczeństwa, aby chronić przed próbami modyfikacji lub atakami ze strony osób działających w złej wierze.


Ten kod wykonuje proces zapytania i generowania odpowiedzi:

`prompt = create_context(query = "Czym są systemy wysokiego ryzyka?", top = 3)` - tworzy kontekst dla zadanego pytania:
- szuka 3 najbardziej podobnych fragmentów tekstu do pytania
- formatuje je jako kontekst dla modelu

`generate(prompt)` - używa przygotowanego kontekstu do wygenerowania odpowiedzi przez model językowy:
- przetwarza kontekst i pytanie przez tokenizer
- generuje odpowiedź używając skonfigurowanego wcześniej modelu
- strumieniowo wyświetla odpowiedź
- zapisuje wynik do pliku JSON

Jest to przykład działania systemu RAG (Retrieval Augmented Generation), który:
1. Znajduje odpowiednie fragmenty tekstu
2. Używa ich jako kontekstu
3. Generuje odpowiedź bazując na znalezionych informacjach

In [ ]:
pd.read_json('5a96c75e.json')

,prompt,output
0,"{'role': 'user', 'content': 'Wyłącznie na pods...",<s><|im_start|> user\nWyłącznie na podstawie p...
